In [1]:
import ROOT
import os
import glob

In [2]:
fileDirPath = "/d/home/septian/Moments2Amplitudes/brufit/"

# file = "result_tree_L4_only.root"
file = "result_tree_BWa2_freeSWaves_326.root"

In [3]:
f = ROOT.TFile(os.path.join(fileDirPath, file), "READ")

In [4]:
t = f.Get("result")

In [5]:
df = ROOT.RDataFrame("result", f)
col_names = df.GetColumnNames()

In [6]:

amp_col_names = [col for col in col_names if not col.startswith("aphi_") and not col.startswith("bphi_") and not col.startswith("a_0_") and not col.startswith("b_0_") and not col.startswith("a_1_") and not col.startswith("b_1_") and not col.startswith("H_") and not col.startswith("mass_bin")]
print("Amplitude column names:", amp_col_names)

for col in amp_col_names:
    print(f"Column: {col}")

Amplitude column names: [b'a_2_-1', b'a_2_-2', b'a_2_0', b'a_2_1', b'a_2_2', b'b_2_-1', b'b_2_-2', b'b_2_0', b'b_2_1', b'b_2_2']
Column: a_2_-1
Column: a_2_-2
Column: a_2_0
Column: a_2_1
Column: a_2_2
Column: b_2_-1
Column: b_2_-2
Column: b_2_0
Column: b_2_1
Column: b_2_2


In [7]:
mom_col_names = [col for col in col_names if col.startswith("H_0_4") or col.startswith("H_1_4") or col.startswith("H_2_4")]

print("Moment column names:", mom_col_names)
for col in mom_col_names:
    print(f"Column: {col}")

Moment column names: [b'H_0_4_0', b'H_0_4_1', b'H_0_4_2', b'H_0_4_3', b'H_0_4_4', b'H_1_4_0', b'H_1_4_1', b'H_1_4_2', b'H_1_4_3', b'H_1_4_4', b'H_2_4_1', b'H_2_4_2', b'H_2_4_3', b'H_2_4_4']
Column: H_0_4_0
Column: H_0_4_1
Column: H_0_4_2
Column: H_0_4_3
Column: H_0_4_4
Column: H_1_4_0
Column: H_1_4_1
Column: H_1_4_2
Column: H_1_4_3
Column: H_1_4_4
Column: H_2_4_1
Column: H_2_4_2
Column: H_2_4_3
Column: H_2_4_4


In [8]:
import sys
sys.path.append("/d/home/septian/EtaPi0Analysis/jupyter_notebook/")
import BruFitResults as FitResults

In [9]:
fit_result_dir = "/d/home/septian/EtaPi0Analysis/run_merged/fitMoment_GlueX1_2019_11_t010100_m080200_MCMCN6000BI1000S08WCOV_R6.34/"

mass_name = "Mpi0eta"
N_mass_bin = 30
mass_bin_width = 0.04
first_mass_bin_center = 0.82

mass_bin = [first_mass_bin_center + mass_bin_width * i for i in range(N_mass_bin)]
mass_bin_dir = [f"{mass_name}{mass:.6f}_/" for mass in mass_bin]

MCMCResults_Phase1_GlueX2020_nominal = FitResults.MomentsMCMC(fit_result_dir, mass_bin_dir, mass_name)

In [10]:
# MCMCResults_Phase1_GlueX2020_nominal.load_rdf()

In [11]:
# MCMCResults_Phase1_GlueX2020_nominal.load_acceptance()

In [12]:
# MCMCResults_Phase1_GlueX2020_nominal.save_unnormalized_moments_vs_mass(fit_result_dir + "Moments_corrected_unnormalized_vs_mass_Phase1_GlueX2020.root")

In [13]:
def MakeAmplitudeSqrVsMass(tree, amp_name):
    gr = ROOT.TGraph()
    gr.SetName(f"{amp_name}_sqr_vs_mass")
    gr.SetTitle(f"{amp_name} squared vs Mass")
    gr.GetXaxis().SetTitle("Mass (GeV/c^{2})")
    gr.GetYaxis().SetTitle(f"{amp_name} squared (GeV/c^{2})^{2}")
    tree.SetBranchStatus("*", 0)  # Disable all branches
    tree.SetBranchStatus("mass_bin", 1)  # Enable only mass_bin branch
    tree.SetBranchStatus(amp_name, 1)  # Enable the specific amplitude branch
    for entry in tree:
        mass = tree.GetLeaf("mass_bin").GetValue()
        amp_value = tree.GetLeaf(amp_name).GetValue()
        gr.SetPoint(gr.GetN(), mass, amp_value*amp_value)
    return gr

def MakeMomentVsMassGraph(tree, moment_name):
    gr = ROOT.TGraph()
    tree.SetBranchStatus("*", 0)
    tree.SetBranchStatus("mass_bin", 1)
    tree.SetBranchStatus(f"{moment_name}", 1)
    for entry in range(tree.GetEntries()):
        tree.GetEntry(entry)
        mass_bin_value = tree.GetLeaf("mass_bin").GetValue()
        moment_value = tree.GetLeaf(f"{moment_name}").GetValue()/2
        if mass_bin_value >= 0 and mass_bin_value < len(mass_bin):
            gr.SetPoint(gr.GetN(), mass_bin_value, moment_value)
    
    return gr

In [14]:
gr_H_4 = []
for colname in mom_col_names:
    # print(f"Processing column: {colname}")
    gr = MCMCResults_Phase1_GlueX2020_nominal.load_saved_unnormalized_moments_vs_mass(fit_result_dir + "Moments_corrected_unnormalized_vs_mass_Phase1_GlueX2020.root", colname)
    gr_H_4.append(gr)

In [15]:
len(gr_H_4)

14

In [16]:
gr_H_4_mom2amps = []
for colname in mom_col_names:
    gr = MakeMomentVsMassGraph(t, colname)
    gr_H_4_mom2amps.append(gr)

In [17]:
len(gr_H_4_mom2amps)

14

In [18]:
out_pdf_dir = "/d/home/septian/EtaPi0Plot/"

In [19]:
canvas = ROOT.TCanvas("canvas", "Moments vs Mass", 1800, 600)

canvas.Print(out_pdf_dir + "Moments_vs_Mass_H_4_free_Swaves.pdf[")

ROOT.gStyle.SetOptTitle(1)

for j in range(5):
    canvas.Clear()
    canvas.Divide(3, 1)
    for i in range(3):
        if i + 3*j >= len(gr_H_4):
            continue
        canvas.cd(i+1)
        gr_H_4[3*j+i].DrawClone("AP")
        # gr_H_4_mom2amps[i].SetMarkerStyle(2)
        # gr_H_4_mom2amps[i].SetMarkerColor(2)
        gr_H_4_mom2amps[3*j+i].SetLineColor(2)
        gr_H_4_mom2amps[3*j+i].SetLineWidth(2)
        gr_H_4_mom2amps[3*j+i].DrawClone("C same")

        # legend = ROOT.TLegend(0.5, 0.7, 0.9, 0.9)
        # legend.AddEntry(gr_H_4[3*j+i], "Unnorm. moments", "pl")
        # legend.AddEntry(gr_H_4_mom2amps[3*j+i], "Fit", "l")
        # legend.SetTextSize(0.035)
        # legend.DrawClone("same")
    
    canvas.Update()
    canvas.Print(out_pdf_dir + f"Moments_vs_Mass_H_4_free_Swaves.pdf")

canvas.Print(out_pdf_dir + "Moments_vs_Mass_H_4_free_Swaves.pdf]")

Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_4_free_Swaves.pdf has been created
Info in <TCanvas::Print>: Current canvas added to pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_4_free_Swaves.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_4_free_Swaves.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_4_free_Swaves.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_4_free_Swaves.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_4_free_Swaves.pdf
Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_4_free_Swaves.pdf has been closed


In [20]:
gr_H000 = MCMCResults_Phase1_GlueX2020_nominal.load_saved_unnormalized_moments_vs_mass(fit_result_dir + "Moments_corrected_unnormalized_vs_mass_Phase1_GlueX2020.root", "H_0_0_0")

In [21]:
canvas_H000 = ROOT.TCanvas("canvas_H000", "H_0_0_0 vs Mass", 1200, 1200)

gr_H000.Draw("AP")
gr_H000.SetMarkerStyle(20)
gr_H000.SetMarkerSize(1.5)
gr_H000.SetLineWidth(2)
canvas_H000.Update()
canvas_H000.Draw()
canvas_H000.Print(out_pdf_dir + "H_0_0_0_vs_Mass.pdf")

Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/H_0_0_0_vs_Mass.pdf has been created


In [22]:
gr_amp_vs_mass = {}
tree = t
chi2 = f.Get("chi2").GetLeaf("chi2").GetValue()
for amp_name in amp_col_names:
    print(f"Processing amplitude: {amp_name}")
    gr = MakeAmplitudeSqrVsMass(tree, f"{amp_name}")
    gr_amp_vs_mass[f"{amp_name}"] = [chi2, gr]

Processing amplitude: a_2_-1
Processing amplitude: a_2_-2
Processing amplitude: a_2_0
Processing amplitude: a_2_1
Processing amplitude: a_2_2
Processing amplitude: b_2_-1
Processing amplitude: b_2_-2
Processing amplitude: b_2_0
Processing amplitude: b_2_1
Processing amplitude: b_2_2


In [23]:
canvas_a_2 = ROOT.TCanvas("canvas_a_2", "D-waves positive refl amp. squared vs Mass", 3000, 600)
ROOT.gStyle.SetOptTitle(0)

refl_dict = {"a": "+", "b": "-"}
l_dict = {"0": "S", "1": "P", "2": "D", "3": "F", "4": "G"}

col_a_2_names = [col for col in amp_col_names if col.startswith("a_2")]
canvas_a_2.Divide(len(col_a_2_names), 1)
for j,name in enumerate(col_a_2_names):
    canvas_a_2.cd(j+1)
    gr_H000.Draw("AP")  # Draw the H_0_0_0 graph with axis
    gr_H000.GetXaxis().SetRangeUser(0.8, 1.6)
    gr_H000.SetMinimum(0.0)
    gr_H000.SetLineWidth(2)

    ROOT.gPad.SetLeftMargin(0.15)
    ROOT.gPad.SetRightMargin(0.0)
    refl,l,m = name.split("_")
    legend = ROOT.TLegend(0.75, 0.7, 1.0, 0.9)

    legend.AddEntry(gr_H000, "H_{0}(0,0)", "lpe")
    legend.SetTextSize(0.06)
    legend.SetBorderSize(0)
    legend.SetFillColorAlpha(0, 0.0)
    
    # canvas_a_2_2.cd(i + 1)
    chi2, gr = gr_amp_vs_mass[f"{name}"]
    # gr.SetMarkerColor(i + 1)
    gr.SetLineColorAlpha(12, 0.2)
    gr.SetLineWidth(2)
    gr.SetFillColorAlpha(12, 0.2)
    # gr.SetMarkerStyle(20 + i)
    # gr.SetTitle(f"a_2_2 vs Mass (Chi2: {chi2})")
    gr.SetTitle("#chi^{2} = " + f"{chi2:.2f}")
    gr.GetXaxis().SetTitle("M_{#eta#pi^{0}} (GeV/c^{2})")
    gr.GetYaxis().SetTitle(f"|{l_dict[l]}^{{{refl_dict[refl]}}}_{{{m}}}|^{{{2}}}")
    # if i == 0:
    #     gr.Draw("APC")  # Draw the first graph with axis
    # else:
        # gr.DrawClone("PC same")
    gr.DrawClone("CF same")
    legend.AddEntry(gr, f"{l_dict[l]}^{{{refl_dict[refl]}}}_{{{m}}}", "l")
    legend.DrawClone()

# canvas_a_2_2.BuildLegend()
canvas_a_2.Draw()
canvas_a_2.Print(out_pdf_dir + "Mom2Amps_a_2_vs_mass_l4Only_model.pdf")

Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/Mom2Amps_a_2_vs_mass_l4Only_model.pdf has been created


In [24]:
col_b_2_names = [col for col in amp_col_names if col.startswith("b_2")]

canvas_b_2 = ROOT.TCanvas("canvas_b_2", "D-waves negative refl amp. squared vs Mass", 3000, 600)
canvas_b_2.Divide(len(col_b_2_names), 1)
for j,name in enumerate(col_b_2_names):
    canvas_b_2.cd(j+1)
    gr_H000.Draw("AP")  # Draw the H_0_0_0 graph with axis
    gr_H000.GetXaxis().SetRangeUser(0.8, 1.6)
    gr_H000.SetMinimum(0.0)
    gr_H000.SetLineWidth(2)

    ROOT.gPad.SetLeftMargin(0.15)
    ROOT.gPad.SetRightMargin(0.0)
    refl,l,m = name.split("_")
    legend = ROOT.TLegend(0.75, 0.7, 1.0, 0.9)

    legend.AddEntry(gr_H000, "H_{0}(0,0)", "lpe")
    legend.SetTextSize(0.06)
    legend.SetBorderSize(0)
    legend.SetFillColorAlpha(0, 0.0)
    
    chi2, gr = gr_amp_vs_mass[f"{name}"]
    gr.SetLineColorAlpha(12, 0.2)
    gr.SetLineWidth(2)
    gr.SetFillColorAlpha(12, 0.2)
    gr.SetTitle("#chi^{2} = " + f"{chi2:.2f}")
    gr.GetXaxis().SetTitle("M_{#eta#pi^{0}} (GeV/c^{2})")
    gr.GetYaxis().SetTitle(f"|{l_dict[l]}^{{{refl_dict[refl]}}}_{{{m}}}|^{{{2}}}")
    gr.DrawClone("CF same")
    legend.AddEntry(gr, f"{l_dict[l]}^{{{refl_dict[refl]}}}_{{{m}}}", "l")
    legend.DrawClone()


canvas_b_2.Draw()
canvas_b_2.Print(out_pdf_dir + "Mom2Amps_b_2_vs_mass_l4Only_model.pdf")

Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/Mom2Amps_b_2_vs_mass_l4Only_model.pdf has been created


In [25]:
mom_col_names = [col for col in col_names if col.startswith("H_0_2") or col.startswith("H_1_2") or col.startswith("H_2_2")]

print("Moment column names:", mom_col_names)
for col in mom_col_names:
    print(f"Column: {col}")

Moment column names: [b'H_0_2_0', b'H_0_2_1', b'H_0_2_2', b'H_1_2_0', b'H_1_2_1', b'H_1_2_2', b'H_2_2_1', b'H_2_2_2']
Column: H_0_2_0
Column: H_0_2_1
Column: H_0_2_2
Column: H_1_2_0
Column: H_1_2_1
Column: H_1_2_2
Column: H_2_2_1
Column: H_2_2_2


In [26]:
gr_H_2 = []
for colname in mom_col_names:
    # print(f"Processing column: {colname}")
    gr = MCMCResults_Phase1_GlueX2020_nominal.load_saved_unnormalized_moments_vs_mass(fit_result_dir + "Moments_corrected_unnormalized_vs_mass_Phase1_GlueX2020.root", colname)
    gr_H_2.append(gr)

In [27]:
gr_H_2_mom2amps = []
for colname in mom_col_names:
    gr = MakeMomentVsMassGraph(t, colname)
    gr_H_2_mom2amps.append(gr)

In [28]:
ROOT.gStyle.SetOptTitle(1)
canvas_H_2 = ROOT.TCanvas("canvas_H_2", "Moments vs Mass H_2", 2000, 1000)
canvas_H_2.Divide(4, 2)
for j in range(len(gr_H_2_mom2amps)):
    canvas_H_2.cd(j + 1)
    gr_H_2[j].DrawClone("AP")
    gr_H_2[j].SetMarkerStyle(20)
    gr_H_2[j].SetMarkerSize(1.0)
    gr_H_2[j].SetLineWidth(2)
    gr_H_2[j].SetLineColor(1)
    gr_H_2_mom2amps[j].SetMarkerColor(2)
    gr_H_2_mom2amps[j].SetLineColor(2)
    gr_H_2_mom2amps[j].SetLineWidth(2)
    gr_H_2_mom2amps[j].DrawClone("C same")

canvas_H_2.Update()
canvas_H_2.Draw()

In [29]:
H2_MultipleFitFileName = "result_tree_BWa2_freeSWaves_[].root"
H2_MultipleFitFileNames = [H2_MultipleFitFileName.replace("[]", str(i)) for i in range(500)]
H2_MultipleFitFileNames = [os.path.join(fileDirPath, name) for name in H2_MultipleFitFileNames]

for file in H2_MultipleFitFileNames:
    print(f"Found file: {file}")

Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_0.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_1.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_2.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_3.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_4.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_5.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_6.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_7.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_8.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_9.root
Found file: /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_10.root
Found file: /d/home/

In [30]:
H2_files = [ROOT.TFile(file, "READ") for file in H2_MultipleFitFileNames if os.path.exists(file)]

In [31]:
def GetChi2FromFile(tFile):
    chi2_tree = tFile.Get("chi2")
    if chi2_tree:
        chi2_tree.GetEntry(0)
        chi2_value = chi2_tree.GetLeaf("chi2").GetValue()
        return chi2_value
    else:
        print(f"No chi2 tree found in {tFile.GetName()}")
        return None

In [32]:
chi2_values = {}
for tFile in H2_files:
    chi2_value = GetChi2FromFile(tFile)
    if chi2_value is not None:
        chi2_values[tFile.GetName()] = chi2_value
        print(f"Chi2 value for {tFile.GetName()}: {chi2_value:.2f}")

Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_0.root: 4600.40
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_1.root: 4600.40
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_2.root: 4649.98
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_3.root: 4601.60
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_4.root: 4617.87
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_5.root: 4600.39
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_6.root: 4600.39
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_7.root: 4600.40
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_8.root: 4602.35
Chi2 value for /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_9.root

In [33]:
# Plotting chi2 values
canvas_chi2 = ROOT.TCanvas("canvas_chi2", "Chi2 Values", 1200, 800)
# canvas_chi2.SetGrid()
h1_chi2 = ROOT.TH1D("h1_chi2", "Chi2 Values", 5001, 999.5, 6000.5)

for i, (file_name, chi2_value) in enumerate(chi2_values.items()):
    h1_chi2.Fill(chi2_value)
    # print(f"Chi2 value for {file_name}: {chi2_value:.2f}")

canvas_chi2.cd()
h1_chi2.SetFillColor(ROOT.kBlue)
h1_chi2.SetLineColor(ROOT.kBlue)
# h1_chi2.SetTitle("Chi2 Values Distribution")
h1_chi2.GetXaxis().SetTitle("#chi^{2} value")
h1_chi2.GetYaxis().SetTitle("Counts")
h1_chi2.Draw("HIST")
canvas_chi2.Update()
canvas_chi2.Draw()
canvas_chi2.Print(out_pdf_dir + "H2_fit_chi2_Values_Distribution.pdf")

Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/H2_fit_chi2_Values_Distribution.pdf has been created


In [34]:
ROOT.gStyle.SetOptStat(0)
# canvas_chi2.Draw()

In [35]:
# take 20 lowest chi2 values
sorted_chi2 = sorted(chi2_values.items(), key=lambda x: x[1])
lowest_chi2_files = sorted_chi2[:20]
print("20 files with lowest chi2 values:")
for file_name, chi2_value in lowest_chi2_files:
    print(f"{file_name}: {chi2_value:.2f}")

20 files with lowest chi2 values:
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_326.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_431.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_317.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_296.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_412.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_309.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_432.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_486.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_300.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_339.root: 1644.98
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_404.root: 1644.9

In [36]:
lowest_chi2_trees = []
for file_name, chi2_value in lowest_chi2_files:
    tFile = ROOT.TFile(file_name, "READ")
    if tFile:
        tree = tFile.Get("result")
        if tree:
            lowest_chi2_trees.append(tree)
            print(f"Loaded tree from {file_name}")

Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_326.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_431.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_317.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_296.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_412.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_309.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_432.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_486.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_300.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_339.root
Loaded tree from /d/home/septian/Moments

In [37]:
gr_H_2_lowest_chi2 = []
for tree in lowest_chi2_trees:
    gr_chi2 = []
    for colname in mom_col_names:
        gr = MakeMomentVsMassGraph(tree, colname)
        gr_chi2.append(gr)
    gr_H_2_lowest_chi2.append(gr_chi2)

In [38]:
ROOT.gStyle.SetOptTitle(1)
canvas_H_2 = ROOT.TCanvas("canvas_H_2", "Moments vs Mass H_2", 2000, 1000)
canvas_H_2.Divide(4, 2)
for j in range(len(gr_H_2)):
    canvas_H_2.cd(j + 1)
    gr_H_2[j].DrawClone("AP")
    gr_H_2[j].SetMarkerStyle(20)
    gr_H_2[j].SetMarkerSize(1.0)
    gr_H_2[j].SetLineWidth(2)
    gr_H_2[j].SetLineColor(1)
    for gr_H_2_mom2amps in gr_H_2_lowest_chi2:
        # gr_H_2_mom2amps[j].SetMarkerColor(2)
        gr_H_2_mom2amps[j].SetLineColorAlpha(2,0.2)
        gr_H_2_mom2amps[j].SetLineWidth(2)
        gr_H_2_mom2amps[j].DrawClone("C same")

canvas_H_2.Update()
canvas_H_2.Draw()
canvas_H_2.Print(out_pdf_dir + "Moments_vs_Mass_H_2_lowest_chi2_freeSwaves_freeDwaves.pdf")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_H_2
Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_2_lowest_chi2_freeSwaves_freeDwaves.pdf has been created


In [39]:
# take 20 lowest chi2 values that are more than 2000
lowest_chi2_files = [item for item in sorted_chi2 if item[1] > 2000]
lowest_chi2_files = lowest_chi2_files[:20]

for file_name, chi2_value in lowest_chi2_files:
    print(f"{file_name}: {chi2_value:.2f}")

/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_345.root: 4189.67
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_239.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_55.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_40.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_209.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_159.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_132.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_241.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_225.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_20.root: 4600.39
/d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_23.root: 4600.39
/d/home/septian/Moments2Amplitudes/b

In [40]:
lowest_chi2_trees = []
for file_name, chi2_value in lowest_chi2_files:
    tFile = ROOT.TFile(file_name, "READ")
    if tFile:
        tree = tFile.Get("result")
        if tree:
            lowest_chi2_trees.append(tree)
            print(f"Loaded tree from {file_name}")

Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_345.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_239.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_55.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_40.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_209.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_159.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_132.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_241.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_225.root
Loaded tree from /d/home/septian/Moments2Amplitudes/brufit/result_tree_BWa2_freeSWaves_20.root
Loaded tree from /d/home/septian/Moments2Am

In [41]:
gr_H_2_lowest_chi2 = []
for tree in lowest_chi2_trees:
    gr_chi2 = []
    for colname in mom_col_names:
        gr = MakeMomentVsMassGraph(tree, colname)
        gr_chi2.append(gr)
    gr_H_2_lowest_chi2.append(gr_chi2)

In [42]:
canvas_H_2.Clear()
canvas_H_2.Divide(4, 2)
for j in range(len(gr_H_2)):
    canvas_H_2.cd(j + 1)
    gr_H_2[j].DrawClone("AP")
    gr_H_2[j].SetMarkerStyle(20)
    gr_H_2[j].SetMarkerSize(1.0)
    gr_H_2[j].SetLineWidth(2)
    gr_H_2[j].SetLineColor(1)
    for gr_H_2_mom2amps in gr_H_2_lowest_chi2:
        # gr_H_2_mom2amps[j].SetMarkerColor(2)
        gr_H_2_mom2amps[j].SetLineColorAlpha(2,0.1)
        gr_H_2_mom2amps[j].SetLineWidth(2)
        gr_H_2_mom2amps[j].DrawClone("C same")

canvas_H_2.Update()
canvas_H_2.Draw()
canvas_H_2.Print(out_pdf_dir + "Moments_vs_Mass_H_2_lowest_chi2_freeSwaves_fixedDwaves.pdf")

Info in <TCanvas::Print>: pdf file /d/home/septian/EtaPi0Plot/Moments_vs_Mass_H_2_lowest_chi2_freeSwaves_fixedDwaves.pdf has been created
